# Homework 1 — Frequent Pattern Analysis

> **Study/reference notebook:** The assignment requires you to write your own code. Study this approach and implement it in the original Coursera notebook. Preserve its filename, cells, metadata, function signatures, and data paths when submitting.

This assignment covers Apriori frequent itemset mining and FP-tree construction using transaction data at `./data/dataset.pickle`.

## Chapter 1 concepts

An itemset's support count is the number of transactions containing all its items; relative support is that count divided by the number of transactions. An itemset is frequent when relative support reaches `min_support`. Apriori uses downward closure: every subset of a frequent set must also be frequent, so candidates with any infrequent subset can be pruned. FP-Growth counts and orders items, removes infrequent items, then inserts transactions into a prefix tree; shared prefixes reuse nodes and increment counts.

## Assignment rules and imports

Do not install packages; preserve the provided APIs and relative data path. The pickle data and grader are not included in this reference notebook.

In [ ]:
import argparse
import pandas as pd
import numpy as np
import random
import pickle5 as pickle
from pathlib import Path
from collections import defaultdict
import itertools

## Supplied helper functions

Leave these helper functions unchanged in the original assignment notebook.

In [ ]:
def findsubsets(s, n):
    subsets = list(sorted(itertools.combinations(s, n)))
    return subsets

def items_from_frequent_itemsets(frequent_itemset):
    items = list()
    for keys in frequent_itemset.keys():
        for item in list(keys):
            items.append(item)
    return sorted(list(set(items)))

## Problem 1 — Apriori implementation (10 points)

Implement `generate_frequent_itemsets(dataset, support, items, n=1, frequent_items={})`. Return singleton keys as item strings and larger itemsets as sorted tuples, with support counts as values. For size 1, count each item. For larger sizes, generate candidates, prune those with an infrequent (n−1)-subset, count the remaining candidates, and keep those meeting the support threshold.

In [ ]:
def generate_frequent_itemsets(dataset, support, items, n=1, frequent_items={}):
    len_transactions = len(dataset)
    frequent_itemsets = dict()
    if len_transactions == 0:
        return frequent_itemsets

    if n == 1:
        for item in items:
            count = sum(item in transaction for transaction in dataset.values())
            if count / len_transactions >= support:
                frequent_itemsets[item] = count
    else:
        candidates = findsubsets(sorted(set(items)), n)
        for candidate in candidates:
            subsets = findsubsets(candidate, n - 1)
            if n == 2:
                all_subsets_frequent = all(s[0] in frequent_items for s in subsets)
            else:
                all_subsets_frequent = all(tuple(s) in frequent_items for s in subsets)
            if not all_subsets_frequent:
                continue
            count = sum(set(candidate).issubset(set(transaction)) for transaction in dataset.values())
            if count / len_transactions >= support:
                frequent_itemsets[tuple(candidate)] = count
    return frequent_itemsets

The prompt's visible example uses five transactions and minimum support `0.5`; it expects frequent singletons `A: 3`, `B: 5`, `D: 4`, pairs `('A', 'B'): 3`, `('B', 'D'): 4`, and no frequent triples. Run the provided tests in Coursera to check the implementation.

## Problem 2 — FP-Growth preparation (10 points)

The required functions count frequent items, prune and reorder each transaction, and build an FP-tree represented as nested dictionaries. The prompt does not ask for conditional pattern bases or pattern extraction.

In [ ]:
def item_support(dataset, min_support):
    len_transactions = len(dataset)
    counts = defaultdict(int)
    for key, value in dataset.items():
        # An item contributes at most once to the count for each transaction.
        for item in set(value):
            counts[item] += 1
    support_dict = dict(sorted(counts.items(), key=lambda item: item[1], reverse=True))
    return support_dict

In [ ]:
def reorder_transactions(dataset, min_support):
    item_counts = item_support(dataset, min_support)
    len_transactions = len(dataset)
    pruned_support = {item: count for item, count in item_counts.items()
                      if len_transactions and count / len_transactions >= min_support}
    updated_dataset = dict()
    for key, value in dataset.items():
        frequent_values = [item for item in value if item in pruned_support]
        updated_dataset[key] = sorted(frequent_values, key=pruned_support.get, reverse=True)
    return updated_dataset

In [ ]:
def build_fp_tree(updated_dataset):
    fp_tree = dict()
    for key, value in updated_dataset.items():
        current_level = fp_tree
        for item in value:
            if item not in current_level:
                current_level[item] = {'count': 0, 'children': dict()}
            current_level[item]['count'] += 1
            current_level = current_level[item]['children']
    return fp_tree

## Validation and submission

The assignment's visible tests check support counts, descending-support transaction order after pruning, and the nested FP-tree counts. Run those cells in the original Coursera environment; its pickle dataset and hidden tests are not included here. Submit through the original notebook opened with **Work in browser** and copy only code into the designated answer cells. Do not submit this reference notebook instead.